In [1]:
# 1. 必要なライブラリのインストール
!pip install umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew
import umap
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H3_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
N_SAMPLES = 400
K_NEIGHBORS = 15 # 局所的信頼性とハブネスを測る近傍数
results = []

print("=== H3計算実験（偽の引力とアーティファクトの生成）を開始します ===")

# ハブネス（K-最近傍の被参照回数）を計算する関数
def calculate_hubness_skewness(X, k=15):
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
    nn.fit(X)
    # 自身を除くK個の近傍インデックスを取得
    indices = nn.kneighbors(X, return_distance=False)[:, 1:]
    # 各データ点が他のデータの近傍として何回選ばれたかを集計
    counts = np.bincount(indices.flatten(), minlength=X.shape[0])
    # 参照回数の偏り（歪度）を計算
    return skew(counts)

# 3. メインループ（6分割スケール）
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(2, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行回数: {total_trials:,} (N={N_SAMPLES}, D={D_features})")

    # --- A. 初期値依存性を排除したベースライン生成 ---
    np.random.seed(42 + step)
    X_base = np.random.uniform(0, 0.2, size=(N_SAMPLES, D_features)) + 1e-10

    # --- B. AHP的偏り・過分散の注入（ハブの生成） ---
    X_skewed = X_base.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)
    X_skewed[:, :hub_dims] *= skew_multiplier

    # --- C. UMAPによる次元圧縮 ---
    reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42)
    umap_base = reducer.fit_transform(X_base)
    umap_skewed = reducer.fit_transform(X_skewed)

    # --- D. キャリブレーション指標1：Hubness Skewness（ハブの発生度） ---
    hub_skew_base = calculate_hubness_skewness(X_base, k=K_NEIGHBORS)
    hub_skew_skewed = calculate_hubness_skewness(X_skewed, k=K_NEIGHBORS)

    # --- E. キャリブレーション指標2：Trustworthiness（偽の引力・アーティファクトの混入度） ---
    # 値が1.0に近いほど高次元の近傍関係を正しく保持。下がると「偽の隣人」が混入。
    trust_base = trustworthiness(X_base, umap_base, n_neighbors=K_NEIGHBORS, metric='cosine')
    trust_skewed = trustworthiness(X_skewed, umap_skewed, n_neighbors=K_NEIGHBORS, metric='cosine')

    # 結果の記録
    results.append({
        'Step': step,
        'Total_Trials': total_trials,
        'Dimensions': D_features,
        'Hubness_Skewness_Base': hub_skew_base,
        'Hubness_Skewness_Skewed': hub_skew_skewed,
        'Trustworthiness_Base': trust_base,
        'Trustworthiness_Skewed': trust_skewed
    })

    # --- F. グラフ出力（都度保存） ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Step {step}: Trials={total_trials:,} (D={D_features})', fontsize=16)

    # 上段: UMAP散布図
    axes[0,0].scatter(umap_base[:,0], umap_base[:,1], s=10, alpha=0.6)
    axes[0,0].set_title(f'UMAP: Baseline (Trustworthiness: {trust_base:.3f})')
    axes[0,1].scatter(umap_skewed[:,0], umap_skewed[:,1], s=10, alpha=0.6, color='red')
    axes[0,1].set_title(f'UMAP: Skewed (Trustworthiness: {trust_skewed:.3f})')

    # 下段: ハブネス（被参照回数）のヒストグラム（ブラックホールの可視化）
    nn_base = NearestNeighbors(n_neighbors=K_NEIGHBORS+1, metric='cosine').fit(X_base)
    counts_base = np.bincount(nn_base.kneighbors(X_base, return_distance=False)[:, 1:].flatten(), minlength=N_SAMPLES)
    axes[1,0].hist(counts_base, bins=20, color='blue', alpha=0.7)
    axes[1,0].set_title(f'K-NN Occurrence (Base) Skew: {hub_skew_base:.2f}')
    axes[1,0].set_xlabel('Times chosen as a neighbor')

    nn_skewed = NearestNeighbors(n_neighbors=K_NEIGHBORS+1, metric='cosine').fit(X_skewed)
    counts_skewed = np.bincount(nn_skewed.kneighbors(X_skewed, return_distance=False)[:, 1:].flatten(), minlength=N_SAMPLES)
    axes[1,1].hist(counts_skewed, bins=20, color='red', alpha=0.7)
    axes[1,1].set_title(f'K-NN Occurrence (Skewed) Skew: {hub_skew_skewed:.2f}')
    axes[1,1].set_xlabel('Times chosen as a neighbor')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 結果の集計とまとめグラフの作成
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

# まとめグラフ: Trustworthinessの崩壊とハブネスの増大
fig, ax1 = plt.subplots(figsize=(9, 6))

ax1.set_xlabel('Total Trials (N_samples * D_features, log scale)')
ax1.set_xscale('log')
ax1.set_ylabel('Trustworthiness (Higher is better)', color='blue')
ax1.plot(df_results['Total_Trials'], df_results['Trustworthiness_Base'], marker='o', linestyle='--', color='lightblue', label='Base Trust')
ax1.plot(df_results['Total_Trials'], df_results['Trustworthiness_Skewed'], marker='o', color='blue', label='Skewed Trust (Fake Attraction)')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_ylim(0, 1.0)

ax2 = ax1.twinx()
ax2.set_ylabel('Hubness Skewness (Lower is better)', color='red')
ax2.plot(df_results['Total_Trials'], df_results['Hubness_Skewness_Base'], marker='s', linestyle='--', color='lightcoral', label='Base Hubness')
ax2.plot(df_results['Total_Trials'], df_results['Hubness_Skewness_Skewed'], marker='s', color='red', label='Skewed Hubness (Blackholes)')
ax2.tick_params(axis='y', labelcolor='red')

fig.suptitle('H3: Emergence of Fake Attraction (Hubness) vs Loss of Local Trust', fontsize=14)
fig.tight_layout()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.savefig(f"{OUT_DIR}/summary_h3_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

=== H3計算実験（偽の引力とアーティファクトの生成）を開始します ===

[Step 1/6] 試行回数: 1,000 (N=400, D=2)

[Step 2/6] 試行回数: 10,000 (N=400, D=25)

[Step 3/6] 試行回数: 100,000 (N=400, D=250)

[Step 4/6] 試行回数: 1,000,000 (N=400, D=2500)

[Step 5/6] 試行回数: 5,000,000 (N=400, D=12500)

[Step 6/6] 試行回数: 10,000,000 (N=400, D=25000)

実験完了。全データを H3_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>